# Capstone build --- Chapter 4: Tasks, State, and Actions

The loop in Chapter~1 passed three types between the agent and the environment without naming them: the task the agent was given, the state it carried and the actions it proposed. Chapter~4 makes those types explicit, because the governance, evaluation and audit layers added later all read them. A task is more than a prompt, state is where the agent is rather than what was said, and an action is one of a fixed set of typed possibilities.

## The task is more than a prompt

A `TaskSpec` carries the goal, the inputs the agent acts on, the outputs it is expected to produce and the constraints and validation rules it must satisfy. The complaint agent's task names the customer message as its input and the classification and draft as expected outputs, and can carry validation rules the harness checks a result against.

In [ ]:
from agentlab.core.task import TaskSpec, ValidationRule

task = TaskSpec(
    goal='handle a customer complaint under banking policy',
    inputs={'message': 'I was charged a $35 overdraft fee I did not authorize.'},
    expected_outputs=['classification', 'draft_response'],
    constraints=['no unauthorized fee-waiver promises', 'cite governing policy'],
    validation=[ValidationRule(name='grounded',
                               description='every claim carries evidence')],
)
print('goal      :', task.goal)
print('inputs    :', task.inputs)
print('constraints:', task.constraints)
print('validation:', [r.name for r in task.validation])

## State is where the agent is

`AgentState` is serializable so it can be persisted, replayed and audited. It carries the task, the step counter, the accumulated tool results and a status the loop reads. Because it round-trips through a dict, the audit log can store a hash of the state before each step and the replay machinery can reconstruct the case exactly.

In [ ]:
from agentlab.core.state import AgentState

state = AgentState(task=task)
print('initial status:', state.status, '| step:', state.step)
restored = AgentState.from_dict(state.to_dict())
print('round-trips   :', restored.to_dict() == state.to_dict())

## Actions are a closed, typed set

An agent may only ever return one of four actions: a `ToolCall`, an `AskUser`, a `Finish` or an `Escalate`. The set is a discriminated union keyed on `kind`, so a proposed action can be parsed and dispatched without guessing its shape. Chapter~1 used `ToolCall` and `Finish`; `Escalate` is the action the governance layer and the reasoning check raise when a case must go to a human, and `parse_action` reconstructs any of them from a serialized record.

In [ ]:
from agentlab.core.action import ToolCall, AskUser, Finish, Escalate, parse_action

actions = [
    ToolCall(tool_name='classify_complaint', arguments={'message': task.inputs['message']}),
    AskUser(question='Which account was charged?'),
    Escalate(reason='regulatory risk flagged: UDAAP', context={'flags': ['UDAAP']}),
    Finish(output={'classification': 'complaint'}),
]
for a in actions:
    print(f'{a.kind:11s} -> {parse_action(a.model_dump()).__class__.__name__}')

These three types are the vocabulary the rest of the capstone is written in. The `ComplaintAgent` of Chapter~8 is a function from `AgentState` to one of these `Action` types; the governance harness of Chapter~12 logs each `(state, action, observation)` transition; the trajectory of Chapter~10 is the recorded sequence of them. Chapter~5 fixes the concrete `ToolCall` targets by defining the full typed action space.